[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-hyperparameter.ipynb)

# Hyperparameter Tuning

*AIBits Academy · Machine Learning End To End · Model Optimisation · New*

The previous chapter showed how to measure a model honestly. This one covers how to systematically search for the configuration that measures best — without fooling yourself in the process.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **📋 Real-World Case Study — Crystal Structure Classification (Materials Science)**
>
> A materials-science team classified 5,058 ABO₃ perovskite oxide compounds into four crystal structures (cubic 61.6%, orthorhombic 29.7%, rhombohedral 6.3%, tetragonal 2.5% — a heavily imbalanced target) from ionic-radius, electronegativity, and bond-length features. Cross-validated comparison of seven models showed Random Forest and Decision Tree both hit train scores of 1.0 (perfect memorisation) against validation scores of only 0.90 and 0.84 — a textbook overfitting gap — while XGBoost's much smaller gap (train 0.77 → CV 0.73) made it the safer choice despite the lower raw number. Validation curves across `learning_rate`, `n_estimators`, and `max_depth` confirmed the train/CV gap widened steadily past `max_depth = 4`, and a final GridSearchCV (85 estimators, depth 4, learning_rate 0.12) combined with minority-class oversampling lifted rhombohedral recall from 0.07 to 0.39 and tetragonal recall from 0.25 to 0.45 — the classes that mattered most were exactly the ones accuracy alone had been hiding.

## Parameters vs Hyperparameters

**Parameters** (θ, weights, split thresholds) are *learned* from training data by the fitting algorithm. 
**Hyperparameters** (learning_rate, max_depth, k, C, n_estimators, λ) are *chosen before training* and control how learning happens. They cannot be learned by gradient descent on the training loss — a search procedure with its own validation signal is needed.

## Grid Search — Exhaustive but Expensive

Grid Search evaluates every combination in a Cartesian product of specified hyperparameter values, using cross-validation for each combination:

$$\text{Total fits} = |\text{grid}| \times k_{\text{folds}} \qquad \text{where } |\text{grid}| = \prod_i |\text{values}_i|$$

A grid of 5 values for `n_estimators` × 4 for `max_depth` × 4 for `min_samples_leaf` × 3 for `max_features` = 240 combinations. With 5-fold CV, that's **1,200 model fits** — the cost grows exponentially with the number of tuned hyperparameters.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=2000, n_features=10, random_state=42)

param_grid = {
    'n_estimators': [100,200,300],
    'max_depth': [None,10,20],
    'min_samples_leaf': [1,4],
}
gs = GridSearchCV(RandomForestClassifier(random_state=42), param_grid,
                   cv=5, scoring='roc_auc', n_jobs=-1)
gs.fit(X, y)
print(f"Total fits: {len(gs.cv_results_['params'])} combos × 5 folds = {len(gs.cv_results_['params'])*5}")
print(f"Best score: {gs.best_score_:.4f}   Best params: {gs.best_params_}")

## Random Search — Usually Better Than It Sounds

Bergstra & Bengio (2012) showed that for a fixed compute budget, **randomly sampling** hyperparameter combinations from specified distributions typically finds a better configuration than grid search — because most hyperparameters have far less influence on performance than one or two "important" ones, and a grid wastes many evaluations varying unimportant dimensions in lock-step with important ones.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
    'n_estimators': randint(50, 500),
    'max_depth': randint(3, 30),
    'min_samples_leaf': randint(1, 20),
    'max_features': uniform(0.3, 0.7),  # continuous range, not fixed choices
}
rs = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_dist,
    n_iter=40, cv=5, scoring='roc_auc', random_state=42, n_jobs=-1)
rs.fit(X, y)
print(f"40 sampled configs (200 fits) vs grid's 90 fits — often reaches a BETTER optimum")
print(f"Best score: {rs.best_score_:.4f}")

## Successive Halving — Spend Budget Where It Matters

`HalvingGridSearchCV` / `HalvingRandomSearchCV` start by evaluating *all* candidates on a small subset of data/resources, then repeatedly discard the worst half and double the resources for survivors. This concentrates compute on the most promising candidates rather than wasting equal budget on obviously-bad ones.

In [ ]:
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingRandomSearchCV

hrs = HalvingRandomSearchCV(RandomForestClassifier(random_state=42), param_dist,
    resource='n_samples', factor=3, cv=5, scoring='roc_auc', random_state=42)
hrs.fit(X, y)
print(f"Best score: {hrs.best_score_:.4f}   (found with far fewer total sample-fits)")

## Bayesian Optimization — Search Informed by Past Results

Grid and Random Search treat every trial independently — they never learn from previous results. Bayesian Optimization builds a **surrogate probabilistic model** (typically a Gaussian Process, using exactly the machinery from the Kernel Methods chapter) of "hyperparameters → validation score", then uses an **acquisition function** to pick the next point that best balances exploring uncertain regions and exploiting known-good ones.

$$\text{Next trial} = \operatorname*{argmax}_{\theta} \ \mathrm{Acquisition}(\theta \mid \text{surrogate model fit on all trials so far})$$

The most common acquisition function, **Expected Improvement (EI)**, quantifies exactly this explore/exploit trade-off in closed form. Given the surrogate GP's predictive mean μ(θ) and standard deviation σ(θ) at a candidate point, and f* the best score observed so far:

$$\mathrm{EI}(\theta) = \sigma(\theta)\left[z\Phi(z) + \varphi(z)\right] \qquad \text{where } z = \dfrac{\mu(\theta) - f^*}{\sigma(\theta)}$$

Φ and φ are the standard normal CDF and PDF. Intuitively: EI is large either when μ(θ) is high (the surrogate genuinely expects a better score — *exploitation*) or when σ(θ) is large (the surrogate is very uncertain here, so there's a real chance of a pleasant surprise — *exploration*), and small when both the predicted mean and the uncertainty are unpromising. This single formula is what lets Bayesian Optimization automatically balance trying new regions against refining known-good ones, without either behaviour being hand-tuned.

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
    }
    model = RandomForestClassifier(**params, random_state=42)
    return cross_val_score(model, X, y, cv=5, scoring='roc_auc').mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40)
print(f"Best score: {study.best_value:.4f}   Best params: {study.best_params}")
# Optuna's Tree-structured Parzen Estimator (TPE) typically reaches
# Random Search's 40-trial score in ~15-20 trials — big savings when each fit is slow

## Choosing a Strategy

| Strategy | Best when | Watch out for |
|---|---|---|
| Grid Search | ≤ 3 hyperparameters, small discrete ranges, need full reproducible coverage | Exponential blow-up; wastes budget on unimportant dimensions |
| Random Search | 4+ hyperparameters, mixed continuous/discrete, limited compute budget | No guarantee of finding the true optimum; still "blind" to past trials |
| Successive Halving | Training is expensive and early performance is a decent proxy for final performance | Can prematurely discard slow-starting-but-eventually-good configs |
| Bayesian Optimization | Each fit is very expensive (deep learning, large ensembles) — every trial should count | Overhead of the surrogate model itself; less parallelisable than Random Search |

> **⚠ Tuning Needs Its Own Nested Validation**
>
> Whichever search strategy you use, the score reported by `best_score_` is itself an *optimistic* estimate — you searched dozens/hundreds of configurations and kept the best-scoring one, so some luck is baked in (multiple-comparisons bias). For a final, honest performance estimate, wrap the entire search inside an **outer** cross-validation loop (Nested CV, introduced in the previous chapter) so the reported number reflects genuine held-out generalisation, not search-selection luck.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · A small grid search

Grid-search the `C` of an SVC over [0.1, 1, 10] with 5-fold CV on the iris data. Store the fitted search in `gs`, the best `C` in `best_C`.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
Xi, yi = load_iris(return_X_y=True)
gs = best_C = None   # TODO


In [ ]:
try:
    check("three candidates", len(gs.cv_results_["params"]) == 3)
    check("best C reported", best_C in (0.1, 1, 10))
    check("score is high", gs.best_score_ > 0.95)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.datasets import load_iris
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
Xi, yi = load_iris(return_X_y=True)
gs = GridSearchCV(SVC(), {"C": [0.1, 1, 10]}, cv=5).fit(Xi, yi)
best_C = gs.best_params_["C"]

```

</details>

### Exercise 2 · Medium · Random search with distributions

Use `RandomizedSearchCV` with `n_iter=12` over `C ~ loguniform(1e-2, 1e2)` and `gamma ~ loguniform(1e-3, 1e1)` for an RBF `SVC` (5-fold, `random_state=0`). Store the search in `rs`.

In [ ]:
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVC
rs = None   # TODO (reuse Xi, yi)


In [ ]:
try:
    check("twelve sampled configs", len(rs.cv_results_["params"]) == 12)
    check("good score", rs.best_score_ > 0.95)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVC
rs = RandomizedSearchCV(SVC(), {"C": loguniform(1e-2, 1e2), "gamma": loguniform(1e-3, 1e1)}, n_iter=12, cv=5, random_state=0).fit(Xi, yi)

```

</details>

### Exercise 3 · Stretch · Bayesian optimisation with Optuna

Minimise `f(x) = (x - 2)**2 + 1` over `x` in [-10, 10] with an Optuna study (`TPESampler(seed=0)`, 25 trials). Store the study in `study` and the best `x` in `best_x`.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = best_x = None   # TODO


In [ ]:
try:
    check("25 trials ran", len(study.trials) == 25)
    check("best x is close to 2", abs(best_x - 2) < 0.5)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
def objective(trial):
    x = trial.suggest_float("x", -10, 10)
    return (x - 2) ** 2 + 1
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=25)
best_x = study.best_params["x"]

```

Each new trial is proposed where earlier trials suggest the objective is low, so it needs far fewer evaluations than a grid.

</details>

---
*Back to the course: **Machine Learning End To End → Hyperparameter Tuning**.*